In this notebook we write code to compute the terms of the Brownian/fractional rough path. First, we simulate paths of $(W^1,W^2,B)$ (with $B$ in general correlated with $W^1,W^2$), and we apply the hybrid scheme to obtain sample paths of $B^H$. Next, we write functions, that take as input the sample paths of $(W^1,W^2,B^H)$ and two times $s < t$, and return the values of $$ (B^H_{st})^k, \quad \int_s^t W^i_{su} \mathrm{d} W^j_u, \quad \int_s^t B^H_{su} \mathrm{d} W^j_u, \quad \int_s^t (B^H_{su})^m(-B^H_u)^k \mathrm{d} W^i_u $$ where for a path, $Y_{st} \coloneqq Y_t - Y_s$. 

We realised that the hybrid scheme does not take Brownian paths as inputs, rather it computes the fBm directly, without making the underlying Bm available. So the function has to be slightly modified so that it can output $B$ in addition to $B^H$. After that, we should write an easy function that takes as input $\rho_1,\rho_2$ and outputs $(W^1,W^2)$ with correlation $\mathrm{d}B\mathrm{d}W^i = \rho_i \mathrm{d} t$.

In [3]:
import numpy as np
import scipy
import scipy.signal as signal
import scipy.integrate as integrate
import matplotlib.pyplot as plt
import scipy.special as special
import time
import scipy.stats
from scipy.optimize import bisect
from scipy.stats import norm
from hybridScheme import hybrid_scheme

In [5]:
H = 0.1
M = 4
grid_points = 1000
kappa = 1
T = 1



After this, here is a TODO list:

- Work out a nice formula for the derivatives, where the vol is a 1-dimensional projection of a 2-dimensional affine system and the dynamics for the price are $\mathrm{d} S = \varphi(v)S\mathrm{d}W$
- Alternatively/additionally, decide to compute the compositions of vector fields either by symbolic or numerical differentiation
- Once we have a way of generating these compositions of vector fields, write a function that simulates the 1-step Euler scheme, and then another function that does this over a grid. This will be our RDE solver.
- Actually, it's an important sanity check to see that the Euler expansion (for a 1-dimensional RDE driven by a fBm, $\mathrm{d} Y = V(Y) \mathrm{d}B^H$) will *not* converge if the expansion is truncated at order less than $\lfloor 1/H \rfloor$. This is to check that we really need all our rough path terms.
- Ask Jack to throw out some coefficients for us
- Calibrate to option prices (and VIX?)

More theory:
- large deviations
- pricing/hedging

For future directions (but not necessarily to do in this paper), if our calibration is a success:

- Lagged approximation (this might help with pricing)
- Market-dependent vol
- Replace/add one of the $W$'s with $W^K$ with $K>1/2$. This is possible to do when $W^K$ and $B^H$ are independent. What about when they are correlated (this is possible when $K = 1/2$, not sure what happens $>1/2$).
- More than one fBm, as long as you don't have Lie brackets between them. Specify this to the linear case, where the condition becomes commutation of matrices.

(There are more TODOs in the overleaf)